# RF-DETR Inconsistency Fix Plan Notebook

This notebook is a **repair workflow** for the RF-DETR Nano experiment on sugarcane leaf disease detection.

## Why this notebook exists
Your original `RF-DETEr.ipynb` contains a **validation inconsistency**:

- **Cell 1** reads `runs/rfdetr_nano/metrics.csv` and reports approximately:
  - `val/mAP@0.50 ≈ 0.8649`
  - `val/mAP@0.50:0.95 ≈ 0.5810`

- **Cell 3** is labeled as **test-set evaluation**, but it actually points to:
  - `valid/images`
  - `valid/labels`

  and reports roughly:
  - `mAP@0.50 ≈ 0.6973`
  - `mAP@0.50:0.95 ≈ 0.4213`

That gap is too large to use safely in an IEEE paper.

## Main causes suspected
1. **Split mix-up**: Cell 3 says “test” but uses the **validation** split.
2. **Mixed evaluation backends**: the later script uses deprecated `supervision` mAP code.
3. **Potential settings mismatch**: checkpoint choice, thresholds, or preprocessing may differ.

## Goal
Create **one source of truth** for RF-DETR validation and test results before writing the paper.

---

## What this notebook will do
1. Audit the current experiment files
2. Verify split paths and counts
3. Read the training-log validation result
4. Re-evaluate validation and test with **one unified pipeline**
5. Save clean metrics for the manuscript

In [ ]:
# Optional: install missing libraries if needed
# Uncomment only if your environment is missing these.
# !pip install -q pandas pyyaml supervision torchmetrics pycocotools opencv-python matplotlib

## 1. Set your experiment paths

Update these paths before running the notebook.

In [ ]:
from pathlib import Path

# ===== EDIT THESE PATHS =====
DATASET_ROOT = Path(r"C:\Users\User\Documents\Sugar Cane Disease.v4i.yolov8")
RUN_ROOT = Path("runs/rfdetr_nano")
METRICS_CSV = RUN_ROOT / "metrics.csv"
BEST_CKPT = RUN_ROOT / "checkpoint_best_ema.pth"   # keep this fixed for all re-evaluations
DATA_YAML = DATASET_ROOT / "data.yaml"

VAL_IMAGES = DATASET_ROOT / "valid" / "images"
VAL_LABELS = DATASET_ROOT / "valid" / "labels"
TEST_IMAGES = DATASET_ROOT / "test" / "images"
TEST_LABELS = DATASET_ROOT / "test" / "labels"

NUM_CLASSES = 4
CONF_THRESH_CM = 0.50
CONF_THRESH_MAP = 0.01
IOU_THRESH_CM = 0.50
OUTPUT_DIR = Path("rfdet_fix_outputs")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

print("Dataset root:", DATASET_ROOT)
print("Run root    :", RUN_ROOT)
print("Best ckpt   :", BEST_CKPT)
print("Output dir  :", OUTPUT_DIR)

## 2. Quick audit of the original inconsistency

This cell documents the exact mismatch found in the old notebook.

In [ ]:
ISSUE_SUMMARY = {
    "training_log_validation": {
        "source": "metrics.csv final validation row",
        "val_mAP50": 0.864886,
        "val_mAP50_95": 0.580952,
        "precision": 0.887999,
        "recall": 0.858189,
    },
    "later_validation_script": {
        "source": "old Cell 3 using valid/images and valid/labels",
        "reported_as": "TEST evaluation (but actually VALIDATION)",
        "val_mAP50": 0.6973,
        "val_mAP50_95": 0.4213,
    },
    "main_risk": "Mixed split naming + deprecated supervision mAP backend + possible setting mismatch"
}

ISSUE_SUMMARY

## 3. Verify files exist

Do not continue until all required files are found.

In [ ]:
required_paths = [
    DATASET_ROOT, RUN_ROOT, METRICS_CSV, BEST_CKPT, DATA_YAML,
    VAL_IMAGES, VAL_LABELS, TEST_IMAGES, TEST_LABELS
]

missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    print("Missing paths:")
    for p in missing:
        print(" -", p)
else:
    print("All required paths found.")

## 4. Inspect dataset metadata and class names

This ensures class IDs are correct before any evaluation.

In [ ]:
import yaml

with open(DATA_YAML, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

class_names = data_cfg["names"]
print("Class names from data.yaml:")
print(class_names)
print("Number of classes declared:", len(class_names))
assert len(class_names) == NUM_CLASSES, "NUM_CLASSES does not match data.yaml"

## 5. Count images and labels in each split

This makes the split usage explicit and helps catch path mistakes.

In [ ]:
def count_files(path, suffixes):
    return len([p for p in path.iterdir() if p.suffix.lower() in suffixes])

split_counts = {
    "valid_images": count_files(VAL_IMAGES, {".jpg", ".jpeg", ".png", ".bmp", ".webp"}),
    "valid_labels": count_files(VAL_LABELS, {".txt"}),
    "test_images": count_files(TEST_IMAGES, {".jpg", ".jpeg", ".png", ".bmp", ".webp"}),
    "test_labels": count_files(TEST_LABELS, {".txt"}),
}

split_counts

## 6. Read the official training-log validation result

This is the **original reference value** from training.  
It is not yet the final paper value until re-evaluation is completed.

In [ ]:
import pandas as pd

metrics = pd.read_csv(METRICS_CSV)
val_rows = metrics.dropna(subset=["val/mAP_50"]).copy()

print("Validation rows found:", len(val_rows))
print("\nLast validation row from metrics.csv:")
display(val_rows.tail(1))

## 7. Freeze the evaluation policy

**Rule:** from this point onward, use only:
- the **same checkpoint**
- the **same split definitions**
- the **same evaluation backend**
- the **same thresholds**

This notebook will produce the values you should use in the paper.

In [ ]:
EVAL_POLICY = {
    "checkpoint": str(BEST_CKPT),
    "validation_split": str(VAL_IMAGES.parent),
    "test_split": str(TEST_IMAGES.parent),
    "conf_threshold_for_map": CONF_THRESH_MAP,
    "conf_threshold_for_confusion_matrix": CONF_THRESH_CM,
    "iou_threshold_for_confusion_matrix": IOU_THRESH_CM,
    "note": "Use one unified evaluation pipeline for both validation and test."
}
EVAL_POLICY

## 8. Load RF-DETR once, consistently

Keep checkpoint loading fixed across all evaluations.

In [ ]:
from rfdetr import RFDETRNano

model = RFDETRNano(pretrain_weights=str(BEST_CKPT), num_classes=NUM_CLASSES)
model.optimize_for_inference()

print("Model loaded from:", BEST_CKPT)

## 9. Use one unified backend for mAP

The old notebook used `supervision.MeanAveragePrecision.benchmark(...)`, which emitted a warning that the deprecated implementation can disagree with `pycocotools`.

Below is a **replacement plan** using `torchmetrics.detection.MeanAveragePrecision(backend="pycocotools")`.

### Important
`model.predict(image, threshold=...)` returns a `supervision.Detections` object in your earlier notebook.  
This cell converts predictions and YOLO labels into the dictionary format expected by TorchMetrics.

In [ ]:
import cv2
import numpy as np
from torchmetrics.detection.mean_ap import MeanAveragePrecision

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def list_images(images_dir):
    return sorted([p for p in images_dir.iterdir() if p.suffix.lower() in IMAGE_SUFFIXES])

def yolo_to_xyxy(line, img_w, img_h):
    cls, x_c, y_c, w, h = map(float, line.strip().split())
    x1 = (x_c - w / 2.0) * img_w
    y1 = (y_c - h / 2.0) * img_h
    x2 = (x_c + w / 2.0) * img_w
    y2 = (y_c + h / 2.0) * img_h
    return int(cls), [x1, y1, x2, y2]

def load_yolo_targets(label_path, img_w, img_h):
    boxes, labels = [], []
    if label_path.exists():
        with open(label_path, "r", encoding="utf-8") as f:
            for raw in f:
                raw = raw.strip()
                if not raw:
                    continue
                cls, box = yolo_to_xyxy(raw, img_w, img_h)
                boxes.append(box)
                labels.append(cls)
    return {
        "boxes": np.array(boxes, dtype=np.float32).reshape(-1, 4),
        "labels": np.array(labels, dtype=np.int64)
    }

def predict_for_map(image_bgr, threshold=CONF_THRESH_MAP):
    det = model.predict(image_bgr, threshold=threshold)
    # Expected fields from supervision.Detections
    boxes = np.array(det.xyxy, dtype=np.float32).reshape(-1, 4)
    scores = np.array(det.confidence, dtype=np.float32).reshape(-1)
    labels = np.array(det.class_id, dtype=np.int64).reshape(-1)
    return {
        "boxes": boxes,
        "scores": scores,
        "labels": labels
    }

print("Helper functions loaded.")

## 10. Re-evaluate the VALIDATION split with the unified mAP backend

This replaces the old Cell 3 metric path.

In [ ]:
def evaluate_split_map(images_dir, labels_dir, threshold=CONF_THRESH_MAP):
    metric = MeanAveragePrecision(box_format="xyxy", backend="pycocotools")
    image_paths = list_images(images_dir)

    for image_path in image_paths:
        image = cv2.imread(str(image_path))
        if image is None:
            print("Warning: could not read", image_path)
            continue

        h, w = image.shape[:2]
        label_path = labels_dir / f"{image_path.stem}.txt"

        preds = predict_for_map(image, threshold=threshold)
        targets = load_yolo_targets(label_path, w, h)

        metric.update(
            [ {
                "boxes": preds["boxes"],
                "scores": preds["scores"],
                "labels": preds["labels"]
            } ],
            [ {
                "boxes": targets["boxes"],
                "labels": targets["labels"]
            } ]
        )

    return metric.compute()

val_map = evaluate_split_map(VAL_IMAGES, VAL_LABELS, threshold=CONF_THRESH_MAP)
val_map

## 11. Re-evaluate the TEST split with the same backend

This is the number that should be compared against YOLO later.

In [ ]:
test_map = evaluate_split_map(TEST_IMAGES, TEST_LABELS, threshold=CONF_THRESH_MAP)
test_map

## 12. Optional: confusion matrix and per-class precision/recall/F1

Use this **only after** the mAP pipeline is fixed.  
You may still use `supervision` for confusion matrices if needed, but keep mAP on the unified backend above.

If you use confusion-matrix-based class metrics, keep:
- the same threshold
- the same split names
- the same checkpoint

In [ ]:
import supervision as sv
import matplotlib.pyplot as plt

def build_sv_dataset(images_dir, labels_dir):
    return sv.DetectionDataset.from_yolo(
        images_directory_path=str(images_dir),
        annotations_directory_path=str(labels_dir),
        data_yaml_path=str(DATA_YAML),
    )

def predict_for_cm(image):
    return model.predict(image, threshold=CONF_THRESH_CM)

def compute_confusion_matrix(images_dir, labels_dir):
    dataset = build_sv_dataset(images_dir, labels_dir)
    cm = sv.ConfusionMatrix.benchmark(dataset=dataset, callback=predict_for_cm)
    return cm

# Example usage:
# val_cm = compute_confusion_matrix(VAL_IMAGES, VAL_LABELS)
# test_cm = compute_confusion_matrix(TEST_IMAGES, TEST_LABELS)

## 13. Save final clean metrics for the paper

These files become your **single source of truth**.

In [ ]:
def summarize_map(metric_output, split_name):
    summary = {
        "split": split_name,
        "mAP_50_95": float(metric_output["map"].item()) if hasattr(metric_output["map"], "item") else float(metric_output["map"]),
        "mAP_50": float(metric_output["map_50"].item()) if hasattr(metric_output["map_50"], "item") else float(metric_output["map_50"]),
        "mAP_75": float(metric_output["map_75"].item()) if hasattr(metric_output["map_75"], "item") else float(metric_output["map_75"]),
        "mar_1": float(metric_output["mar_1"].item()) if hasattr(metric_output["mar_1"], "item") else float(metric_output["mar_1"]),
        "mar_10": float(metric_output["mar_10"].item()) if hasattr(metric_output["mar_10"], "item") else float(metric_output["mar_10"]),
        "mar_100": float(metric_output["mar_100"].item()) if hasattr(metric_output["mar_100"], "item") else float(metric_output["mar_100"]),
    }
    return summary

val_summary = summarize_map(val_map, "validation")
test_summary = summarize_map(test_map, "test")

final_df = pd.DataFrame([val_summary, test_summary])
final_df.to_csv(OUTPUT_DIR / "rfdetr_unified_metrics.csv", index=False)

with open(OUTPUT_DIR / "rfdetr_eval_policy.json", "w", encoding="utf-8") as f:
    json.dump(EVAL_POLICY, f, indent=2)

display(final_df)
print("Saved:", OUTPUT_DIR / "rfdetr_unified_metrics.csv")
print("Saved:", OUTPUT_DIR / "rfdetr_eval_policy.json")

## 14. Paper-safe interpretation template

Use this only after the re-evaluation is complete.

### If the new validation value is close to `metrics.csv`
Then the old Cell 3 result was likely caused by the deprecated mAP backend or split-labeling confusion.

### If the new validation value is still much lower than `metrics.csv`
Then investigate:
1. whether training-log validation used EMA vs non-EMA checkpoint
2. whether image resizing differs at evaluation time
3. whether class mapping differs
4. whether postprocessing differs between internal and external evaluation
5. whether the wrong checkpoint is being loaded

### Rule for the manuscript
Use **only the values exported by this notebook** in final tables and discussion.

## 15. Final checklist before updating the paper

- [ ] Cell 3/Cell 4 split naming problem is corrected
- [ ] One checkpoint is used for all final RF-DETR metrics
- [ ] mAP is computed with one backend only
- [ ] Validation and test are evaluated separately and labeled correctly
- [ ] Final metrics are exported to `rfdet_fix_outputs/rfdetr_unified_metrics.csv`
- [ ] Old mixed numbers are removed from the paper